# Guard Clauses and Early Returns in Python

### 🎯 Objective
Learn how to eliminate nested conditional blocks ("arrow code") and streamline function control flow using **guard clauses** and **early returns**. 

In professional software development, readable code flows vertically rather than drifting rightward through deep indentation. By validating prerequisites and aborting immediately on failure, your business logic remains prominent, flat, and maintainable.


### 📖 1. Concept Demonstration: Arrow Code vs. Guard Clauses

#### The Problem: Arrow Code (Deep Nesting)
When multiple preconditions are tested using nested `if`/`else` branches, the "happy path" ends up trapped inside multiple indentation levels, forcing readers to track branching state across many lines.


In [ ]:
# Antipattern: Deeply nested conditional logic
class MockUser:
    def __init__(self, is_active: bool, has_permissions: bool, balance: float):
        self._active = is_active
        self._perms = has_permissions
        self.balance = balance

    def is_active(self) -> bool:
        return self._active

    def has_permissions(self) -> bool:
        return self._perms


def process_user_nested(user: MockUser) -> str:
    if user.is_active():
        if user.has_permissions():
            if user.balance > 0:
                return f"Success: Processed user with balance ${user.balance:.2f}"
            else:
                return "Error: Insufficient balance"
        else:
            return "Error: Missing permissions"
    else:
        return "Error: Inactive account"


print("Nested result:", process_user_nested(MockUser(True, True, 50.0)))


#### The Solution: Guard Clauses (Bouncer Pattern)
A **guard clause** handles edge cases, preconditions, or invalid inputs at the very top of the function. If a condition is not met, the function exits immediately (`return` or `raise`). Once all guards have passed, the main logic executes cleanly without indentation.


In [ ]:
# Pythonic Best Practice: Guard Clauses with Early Returns
def process_user_guarded(user: MockUser) -> str:
    # Guard 1: Account status check
    if not user.is_active():
        return "Error: Inactive account"

    # Guard 2: Authorization check
    if not user.has_permissions():
        return "Error: Missing permissions"

    # Guard 3: Business rule check
    if user.balance <= 0:
        return "Error: Insufficient balance"

    # Happy path: Clean, flat, and obvious
    return f"Success: Processed user with balance ${user.balance:.2f}"


print("Guarded result:", process_user_guarded(MockUser(True, True, 50.0)))


### 💻 2. Practice Challenges (Your Turn)

#### Challenge 1: User Registration Validator
Refactor user registration validation into a clean, flat function using guard clauses.

**Rules**:
1. `username` must not be empty or only whitespace. If invalid, return `(False, "Username cannot be empty")`.
2. `email` must contain `'@'` and `'.'`. If invalid, return `(False, "Invalid email address")`.
3. `age` must be an integer >= 18. If invalid, return `(False, "User must be at least 18 years old")`.
4. If all validations pass, return `(True, f"Registration approved for {username.strip()}")`.


In [ ]:
def validate_registration(username: str, email: str, age: int) -> tuple[bool, str]:
    """Validates registration requirements using guard clauses."""
    # YOUR CODE HERE
    pass


# Automated Assertions
assert validate_registration("", "user@example.com", 25) == (False, "Username cannot be empty")
assert validate_registration("   ", "user@example.com", 25) == (False, "Username cannot be empty")
assert validate_registration("alice", "invalid_email", 25) == (False, "Invalid email address")
assert validate_registration("alice", "alice@example", 25) == (False, "Invalid email address")
assert validate_registration("alice", "alice@example.com", 16) == (False, "User must be at least 18 years old")
assert validate_registration("alice", "alice@example.com", 18) == (True, "Registration approved for alice")
assert validate_registration("  bob  ", "bob@work.org", 30) == (True, "Registration approved for bob")
print("✅ Challenge 1 passed all test cases!")


<details>
<summary>💡 Hint</summary>

Check each precondition one-by-one with an inverted condition (`if not username.strip(): return ...`), and let the success tuple be the final return statement of the function.
</details>

<details>
<summary>🔍 Solution</summary>

```python
def validate_registration(username: str, email: str, age: int) -> tuple[bool, str]:
    if not username or not username.strip():
        return (False, "Username cannot be empty")

    if "@" not in email or "." not in email:
        return (False, "Invalid email address")

    if age < 18:
        return (False, "User must be at least 18 years old")

    return (True, f"Registration approved for {username.strip()}")
```
</details>


#### Challenge 2: E-Commerce Order Processing
Implement `calculate_order_payout` using guard clauses.

An order dictionary has the structure:
`{"status": str, "items": list[dict], "coupon": str | None, "user_verified": bool}`

**Rules**:
1. If `order["status"] != "PENDING"`, return `{"success": False, "reason": "Order is not pending"}`.
2. If `not order["user_verified"]`, return `{"success": False, "reason": "Unverified user"}`.
3. If `not order.get("items")` (empty list), return `{"success": False, "reason": "Cart is empty"}`.
4. Calculate subtotal = sum of `item["price"] * item["quantity"]` for all items.
5. If `coupon == "SAVE20"`, apply a 20% discount (`subtotal * 0.8`).
6. Return `{"success": True, "total": round(total, 2)}`.


In [ ]:
def calculate_order_payout(order: dict) -> dict:
    """Calculates order total after validating preconditions using guard clauses."""
    # YOUR CODE HERE
    pass


# Automated Assertions
order_invalid_status = {"status": "CANCELLED", "items": [{"price": 10.0, "quantity": 1}], "coupon": None, "user_verified": True}
order_unverified = {"status": "PENDING", "items": [{"price": 10.0, "quantity": 1}], "coupon": None, "user_verified": False}
order_empty = {"status": "PENDING", "items": [], "coupon": None, "user_verified": True}
order_regular = {"status": "PENDING", "items": [{"price": 25.50, "quantity": 2}, {"price": 10.0, "quantity": 1}], "coupon": None, "user_verified": True}
order_discounted = {"status": "PENDING", "items": [{"price": 50.0, "quantity": 2}], "coupon": "SAVE20", "user_verified": True}

assert calculate_order_payout(order_invalid_status) == {"success": False, "reason": "Order is not pending"}
assert calculate_order_payout(order_unverified) == {"success": False, "reason": "Unverified user"}
assert calculate_order_payout(order_empty) == {"success": False, "reason": "Cart is empty"}
assert calculate_order_payout(order_regular) == {"success": True, "total": 61.0}
assert calculate_order_payout(order_discounted) == {"success": True, "total": 80.0}
print("✅ Challenge 2 passed all test cases!")


<details>
<summary>💡 Hint</summary>

Place guards for `"status"`, `"user_verified"`, and empty `"items"` right at the beginning. Then calculate the sum over the items list and conditionally multiply by 0.8 if the coupon matches `"SAVE20"`.
</details>

<details>
<summary>🔍 Solution</summary>

```python
def calculate_order_payout(order: dict) -> dict:
    if order.get("status") != "PENDING":
        return {"success": False, "reason": "Order is not pending"}

    if not order.get("user_verified"):
        return {"success": False, "reason": "Unverified user"}

    items = order.get("items")
    if not items:
        return {"success": False, "reason": "Cart is empty"}

    total = sum(item["price"] * item["quantity"] for item in items)
    if order.get("coupon") == "SAVE20":
        total *= 0.8

    return {"success": True, "total": round(total, 2)}
```
</details>
